<a href="https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/deepgp_sinc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep GP (DSVI) with `gpc`

Doubly stochastic variational deep GP ([CIP-0035](https://github.com/SheffieldML/GPc/blob/main/cip/cip0035.md)); native [deepgp](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/deepgp.md).

Requires **`gpc>=0.1.0a10`** (DeepGP façade).

```python
m = gpc.models.DeepGP(X, Y, layers=2, hidden=1, num_inducing=10, chi=0.5)
m.optimize(iters=100, adam_lr=0.05)
mu, var = m.predict(Xtest)
```

**Notes**

* Adam maximises the ELBO; `objective()` follows the usual GPc sign convention for display.
* Optional `drop_residual=True` for the finite-network (Nyström residual off) bound.
* Probit / ordered finals: `likelihood='probit'` or `'ordered'`.


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    import matplotlib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])


## 1-D sinc-style toy

In [ ]:
rng = np.random.default_rng(0)
X = np.linspace(-4.0, 4.0, 40)[:, None]
Y = np.sin(1.5 * X)/X + 0.1 * rng.normal(size=X.shape)

## Two-layer DeepGP

In [ ]:
m = gpc.models.DeepGP(
    X, Y, layers=2, hidden=1, num_inducing=8, chi=0.5, beta=5.0
)
print(m)
print("approx", m.approx, "layers", m.num_layers)
before = m.elbo()
m.optimize(iters=20000, adam_lr=0.05, seed=1, messages=True)
print("ELBO", before, "→", m.elbo())
mu, var = m.predict(X)
rmse = float(np.sqrt(np.mean((mu - Y) ** 2)))
print("train RMSE", rmse)

## Predictive mean ± 2σ

In [ ]:
Xs = np.linspace(-5, 5, 200)[:, None]
mu_s, var_s = m.predict(Xs)
sd = np.sqrt(np.maximum(var_s, 0.0))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(X, Y, "kx", ms=5, alpha=0.8, label="data")
ax.plot(Xs, mu_s, "C0-", label="predict")
ax.fill_between(
    Xs[:, 0], (mu_s - 2 * sd)[:, 0], (mu_s + 2 * sd)[:, 0], color="C0", alpha=0.2
)
ax.set_title("DeepGP (L=2) on sinc toy")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()